# Greenhouse Gas Emissions Explorer

### 1. Introduction
This dashboard explores trends in greenhouse gas (GHG) emissions by the world's top-20 climate polluting countries using data from the European Commission Joint Research Centre's [Emissions Database for Global Atmospheric Researcher (EDGAR)](https://edgar.jrc.ec.europa.eu/). The dashboard visualizes countries' emissions over time and provides a detailed breakdown of the sectors, sources, and gases that make up those emissions. You can find the finished dashboard at [URL] and the GitHub repo for this project at [URL].

[INSERT SCREENSHOT HERE]

### 2. Library choice

The dashboard was created in Python using [Panel](https://panel.holoviz.org/) and the [HoloViews](https://holoviews.org/index.html) plotting library. Panel is an open-source library for building interactive data dashboards that handles app layout and provides controls for the plots created with HolowViews and the other libraries. I've chosen Panel for this project because it works with a variety of plotting libraries (such as HoloViews and hvPlot) and integrates seamlessly with Jupyter notebooks, making it easy to illustrate the step-by-step process of building the dashboard here in this notebook. The final product in the notebook is served directly, without additional export steps.

Let's start by downloading our libraries and setting our configurations.

In [1]:
import numpy as np
import pandas as pd
import panel as pn
from urllib.request import urlretrieve

import plotly.graph_objs as go
import plotly.io as pio
import matplotlib.pyplot as plt
import matplotlib as mpl
import hvplot.pandas
import warnings
import wbgapi
import world_bank_data as wb
import zipfile
from pathlib import Path

warnings.simplefilter('ignore')
pn.extension('vega', 'plotly', defer_load=True, sizing_mode="stretch_width")
mpl.use('agg')

# Data directory
DATA_DIR_RAW = Path("../data/raw")
DATA_DIR_INTERIM = Path("../data/interim")

### 3. Data
For the dashboard, I'll use the 2026 release of the EDGAR Community GHG emissions database (EDGAR_2026_GHG), available [here](https://edgar.jrc.ec.europa.eu/dataset_ghg2026). The data are provided by the European Union, European Commission, and Joint Research Centre (JRC) under a [Creative Commons Attribution 4.0 International (CC BY 4.0) licence](https://creativecommons.org/licenses/by/4.0/). See the list of sources at the end of this notebook for full citations.

Let's start by downloading EDGAR_2026_GHG from the EC-JRC site and reading it into a Pandas DataFrame. We'll need the annual totals for GHG emissions by sector and country, which are divided into separate files by gas. The gases represent the six main baskets of climate pollutants covered by the [Kyoto Protocol](): carbon dioxide (CO2), methane (CH4), nitrous oxide (N2O), and the so-called F-gases (hydrofluorocarbons, or HFCs, and perfluorocarbons, or PFCs) and sulphur hexafluoride (SF6).  Totals for each gas are weighted by their [global warming potential](https://ec.europa.eu/eurostat/statistics-explained/index.php?title=Glossary:Global-warming_potential_(GWP)) (i.e., their relative potency in trapping heat) and reported in equivalent units of CO2 (CO2e). To get the detailed breakdown of emissions by sector and gas for each of the top-20 countries, we'll need to download and process the data files for each gas.

In [2]:
# Define function for downloading, unzipping, and reading in EDGAR_2026_GHG data
def load_primap_data(url: str, filename: str, data_dir=DATA_DIR_RAW, force_download=False):
    """
    Download the PRIMAP-hist data, read it into Pandas, and clean it
    
    Parameters
    url             :   str, the URL containing the PRIMAP-hist data
    filename        :   str, the name of the CSV file we want
    data_dir        :   str, relative path name for the raw data
    force_download  :   bool, force Python to download the zip file even if it already exists

    Returns
    A Pandas DataFrame with the annual emissions data for all countries, sectors, and gases we want
    """
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)
    
    # Download the CSV file
    filepath = data_dir.joinpath(filename)
    if force_download or not filepath.exists():
        print(f"Downloading {filename}...")
        urlretrieve(url, filepath)
    else:
        print(f"Data file already exists. Using existing file for {filename}")

    # Read the CSV file into Pandas
    df = pd.read_csv(filepath)
    
    # Only use the "HISTCR" data - corresponds with country-reported data series
    df = df[df["scenario (PRIMAP-hist)"] == "HISTCR"]

    # Pollutants - only keep aggregate baskets/groups of Kyoto Protocol gases
    gases = ["CH4", "CO2", "KYOTOGHG (AR5GWP100)", "N2O", "FGASES (AR5GWP100)"]
    df = df[df['entity'].isin(gases)] 
    df['entity'] = df['entity'].replace({
        "CH4": "ch4",
        "CO2": "co2", 
        "KYOTOGHG (AR5GWP100)": "total",
        "N2O": "n2o",
        "FGASES (AR5GWP100)": "fgas"
    })
    
    # Ignore regional groupings (other than the EU)
    exclude_grps = ["ANNEXI", "AOSIS", "BASIC", "EARTH", "LDC", "NONANNEXI", "UMBRELLA"]
    df = df[~df["area (ISO3)"].isin(exclude_grps)]
    df["area (ISO3)"] = df["area (ISO3)"].replace("EU27BX", "EUU") # use the EU's ISO3 code

    # Only keep the columns we need
    df = df.drop(columns=["source", "scenario (PRIMAP-hist)", "provenance"])

    # Rename columns we keep
    df = df.rename(columns={
        "area (ISO3)": "iso3",
        "category (IPCC2006_PRIMAP)": "sector",
        "entity": "gas"
        })

    # Reshape to long format, using only the columns we need
    id_cols = ["iso3", "gas", "unit", "sector"]
    year_cols = [y for y in df.columns if y not in id_cols]
    df = df.melt(id_vars=id_cols, value_vars=year_cols, var_name="year", value_name="emissions")
    df["year"] = df["year"].astype(int)

    # The PRIMAP data go back to 1750 - let's just keep years since 1990
    df = df[df["year"] >= 1990].reset_index(drop=True)
    
    return df


# Load the data
primap_df = load_primap_data(
    url="https://zenodo.org/records/22876287/files/Guetschow_et_al_2026-PRIMAP-hist_v2.8_final_22-Sep-2026.csv?download=1",
    filename="Guetschow_et_al_2026-PRIMAP-hist_v2.8_final_22-Sep-2026.csv"
)

primap_df.head()


Data file already exists. Using existing file for Guetschow_et_al_2026-PRIMAP-hist_v2.8_final_22-Sep-2026.csv


,iso3,gas,unit,sector,year,emissions
0,ABW,ch4,CH4 * gigagram / yr,1,1990,0.038700
1,ABW,ch4,CH4 * gigagram / yr,1.A,1990,0.025400
2,ABW,ch4,CH4 * gigagram / yr,1.B,1990,0.013300
3,ABW,ch4,CH4 * gigagram / yr,1.B.1,1990,0.013200
4,ABW,ch4,CH4 * gigagram / yr,1.B.2,1990,0.000119


The data include estimated emissions for each granular sub-sector. The `sector` codes correspond with the GHG source categories defined by the [Intergovernmental Panel on Climate Change (IPCC)](https://www.ipcc.ch/). The full list of sectors in the PRIMAP-hist data is provided [here](https://zenodo.org/records/22876287). Let's just get the annual totals and sector aggregates.

In [3]:
# Annual totals by country for all pollutants and sectors
mask = (primap_df['sector'] == "M.0.EL") & (primap_df['gas'] == "total")
ghg_tot = primap_df[mask].reset_index(drop=True)

# Annual totals by country for all pollutants and each sector
aggregated_sectors = ['1', '2', '4', '5', 'M.AG']
mask = (primap_df['gas'] == "total") & (primap_df['sector'].isin(aggregated_sectors))
ghg_by_sector = primap_df[mask].reset_index(drop=True)

# Totals by country for each pollutant and all sectors
mask = (primap_df['sector'] == "M.0.EL") & (primap_df['gas'] != "total")
ghg_by_gas = primap_df[mask].reset_index(drop=True)

# For the units, replace physical values with CO2-equivalents
# Note: We'll apply the IPCC AR5 Global Warming Potential (GWP) values
ar5_gwp = {
    'ch4': 28, # each ton of CH4 has the effect of 28 tons of CO2
    'n2o': 265 
}
ghg_by_gas['emissions'] = ghg_by_gas['emissions'] * ghg_by_gas['gas'].map(ar5_gwp).fillna(1)
ghg_by_gas['unit'] = "CO2 * gigagram / yr" # all gases now in units of Gg CO2e

The PRIMAP-hist data cover over 200 countries and regions. Let's focus on the top 10 countries/regions by total emissions in 2025. The number-one country, China (CHN), emitted the equivalent of 15.6 billion tons CO<sub>2</sub> last year. That's about 2.5x the U.S. total. India (IND), the third largest polluter, the equivalent of around 3.9 billion tons CO<sub>2</sub>.

In [4]:
# Identify the top 10 regions by their 2025 emissions
top10_df = (ghg_tot[ghg_tot['year'] == 2025]
    .sort_values("emissions", ascending=False)
    .head(10).reset_index(drop=True))
top10_df

,iso3,gas,unit,sector,year,emissions
0,CHN,total,CO2 * gigagram / yr,M.0.EL,2025,15600000.0
1,USA,total,CO2 * gigagram / yr,M.0.EL,2025,6320000.0
2,IND,total,CO2 * gigagram / yr,M.0.EL,2025,3900000.0
3,EUU,total,CO2 * gigagram / yr,M.0.EL,2025,2990000.0
4,RUS,total,CO2 * gigagram / yr,M.0.EL,2025,2130000.0
5,BRA,total,CO2 * gigagram / yr,M.0.EL,2025,1290000.0
6,IDN,total,CO2 * gigagram / yr,M.0.EL,2025,1230000.0
7,IRN,total,CO2 * gigagram / yr,M.0.EL,2025,1210000.0
8,JPN,total,CO2 * gigagram / yr,M.0.EL,2025,1020000.0
9,SAU,total,CO2 * gigagram / yr,M.0.EL,2025,806000.0


Let's see how the list of top 10 emitters has shifted over the years. I'll make the rankings easier to read by swapping ISO3 codes for country names. Also, because I've included the EU as a region, I'll drop the individual EU27 member states. The EU states are identified in an interim data set I've created and maintained over the years that lists country names, codes, and group memberships.  

In [5]:
# Get the country list CSV
country_list_filepath = DATA_DIR_INTERIM.joinpath("country_list.csv")
country_list_df = pd.read_csv(country_list_filepath)
country_list_df.columns = country_list_df.columns.str.lower()
country_list_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 236 entries, 0 to 235
Data columns (total 8 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   iso3     236 non-null    str    
 1   name     236 non-null    str    
 2   g20      19 non-null     float64
 3   g7       7 non-null      float64
 4   eu27     27 non-null     float64
 5   in_oecd  36 non-null     float64
 6   annex1   42 non-null     float64
 7   paris    192 non-null    float64
dtypes: float64(6), str(2)
memory usage: 14.9 KB


In [6]:
# Get rankings by year, using country names, dropping individual EU member states
years = [1990, 2000, 2000, 2010, 2020, 2025]
rankings_dict = {}
for yr in years:
    merged_df = ghg_tot.merge(country_list_df, how="left", on="iso3")
    merged_df['name'] = np.where(
        merged_df['iso3'] == "EUU",
        "European Union",
        merged_df['name']
    )
    no_eu_members = merged_df[merged_df['eu27'] != 1]
    top10 = (no_eu_members[no_eu_members['year'] == yr]
    .sort_values("emissions", ascending=False)
    .head(10)
    .reset_index(drop=True))
    rankings_dict[str(yr)] = top10["name"].tolist()

rankings_df = pd.DataFrame(rankings_dict)
rankings_df.head(10)

,1990,2000,2010,2020,2025
0,United States of America,United States of America,China,China,China
1,European Union,China,United States of America,United States of America,United States of America
2,China,European Union,European Union,European Union,India
3,Russian Federation,Russian Federation,India,India,European Union
4,Japan,India,Russian Federation,Russian Federation,Russian Federation
5,India,Japan,Japan,Brazil,Brazil
6,Ukraine,Brazil,Brazil,Japan,Indonesia
7,United Kingdom of Great Britain and Northern I...,Canada,"Iran, Islamic Republic of","Iran, Islamic Republic of","Iran, Islamic Republic of"
8,Brazil,United Kingdom of Great Britain and Northern I...,Canada,Indonesia,Japan
9,Canada,Mexico,Mexico,Saudi Arabia,Saudi Arabia


But what about on a per-capita basis? Taking these 10 countries and regions, the picture looks much different. Let's bring in population data from the World Bank and compare. The World Bank data are from the [World Development Indicators](https://datacatalog.worldbank.org/search/dataset/0037712/world-development-indicators), fetched via the `wbgapi` package, and are provided under a [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/) license. I've had(As a side note, I found the `wbgapi` library after discovering that the `pandas-datareader` library shown in one of our lecture videos no longer works.)

In [7]:
'''
# Function to get population data using World Bank's API
def load_wb_data(url, series, filename, data_dir=DATA_DIR_RAW, force_download=False):
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)
    
    # Download the Excel file - CSV file requests tend to hang
    filepath = data_dir.joinpath(filename)
    if force_download or not filepath.exists():
        print(f"Downloading {filename}...")
        urlretrieve(url, filepath)
    else:
        print(f"Data file already exists. Using existing file for {filename}")

    # Read the CSV file into Pandas
    df = pd.read_excel(filepath, sheet_name= , header= )
    df.columns = df.
    
    if force_download or not 
'''
pop_df = (
    wb.get_series("SP.POP.TOTL", id_or_value="id", date="1990:2025")
      .rename("pop")
      .reset_index()
)
# Make column names consistent with the emissions data
pop_df = pop_df.rename(columns={"Country": 'iso3', "Year": 'year'})
pop_df = pop_df.drop(columns='Series')
pop_df["year"] = pop_df["year"].astype(int)
pop_df.head()


,iso3,year,pop
0,AFE,1990,311748681.0
1,AFE,1991,320442961.0
2,AFE,1992,329082707.0
3,AFE,1993,338324002.0
4,AFE,1994,347441809.0


In [8]:
# Per-capita emissions
top10_percap_df = top10_df.copy()
top10_percap_df = top10_percap_df.merge(pop_df, how="inner", on=["iso3", "year"])
top10_percap_df['emissions_percap'] = top10_percap_df['emissions'] / top10_percap_df['pop'] * 1000
top10_percap_df = top10_percap_df.sort_values('emissions_percap', ascending=False)
top10_percap_df['unit'] = "tCO2 / person / yr"
top10_percap_df = top10_percap_df[['iso3', 'unit', 'year', 'emissions_percap']]
top10_percap_df

,iso3,unit,year,emissions_percap
9,SAU,tCO2 / person / yr,2025,21.799364
1,USA,tCO2 / person / yr,2025,18.491164
4,RUS,tCO2 / person / yr,2025,14.841827
7,IRN,tCO2 / person / yr,2025,13.092733
0,CHN,tCO2 / person / yr,2025,11.090691
8,JPN,tCO2 / person / yr,2025,8.268031
3,EUU,tCO2 / person / yr,2025,6.627839
5,BRA,tCO2 / person / yr,2025,6.061677
6,IDN,tCO2 / person / yr,2025,4.304895
2,IND,tCO2 / person / yr,2025,2.664179


### Visualization techniques

Line charts of total and per capita emissions

In [9]:
import pandas as pd
import panel as pn
import holoviews as hv
from bokeh.models import HoverTool, NumeralTickFormatter

pn.extension(sizing_mode="stretch_width")
hv.extension("bokeh")

# ---------------------------------------------------------------- data
OWID_URL = "https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv"
VALUE_COL = "total_ghg_excluding_lucf"          # MtCO2e, excl. land use
COUNTRIES = ["United States", "China", "Saudi Arabia", "India", "Russia",
             "Japan", "Germany", "Indonesia", "Iran", "Brazil"]

# Color follows the country, not its position in the selection, so a
# country keeps its color when you change what's highlighted.
COLORS = {
    "United States": "#2563eb", "China": "#dc2626", "Saudi Arabia": "#059669",
    "India": "#d97706", "Russia": "#7c3aed", "Japan": "#db2777",
    "Germany": "#0891b2", "Indonesia": "#65a30d", "Iran": "#9333ea",
    "Brazil": "#ea580c",
}
GREY, GREY_HOVER = "#cbd5e1", "#475569"


@pn.cache  # download once per server process, not once per visitor
def load_data() -> pd.DataFrame:
    df = pd.read_csv(OWID_URL, usecols=["country", "year", VALUE_COL])
    return (df[df["country"].isin(COUNTRIES) & (df["year"] >= 1990)]
              .dropna(subset=[VALUE_COL])
              .rename(columns={VALUE_COL: "emissions"}))


df = load_data()
FIRST, LAST = int(df["year"].min()), int(df["year"].max())   # data ends ~2023

# ---------------------------------------------------------------- widgets
highlight = pn.widgets.MultiChoice(
    name="Highlight countries", options=COUNTRIES,
    value=["United States", "China", "Saudi Arabia"], max_items=4)
years = pn.widgets.IntRangeSlider(
    name="Years", start=FIRST, end=LAST, value=(FIRST, LAST), step=1)
log_y = pn.widgets.Switch(name="Log scale", value=False)


# ---------------------------------------------------------------- chart
def _minimal(plot, element):
    p = plot.state
    p.outline_line_color = None
    p.title.text_font_style = "normal"
    for ax in p.axis:
        ax.axis_line_color = None
        ax.major_tick_line_color = None
        ax.minor_tick_line_color = None
        ax.major_label_text_color = "#64748b"
    p.yaxis.formatter = NumeralTickFormatter(format="0,0")


def make_chart(highlighted, year_range, log):
    lo, hi = year_range
    d_all = df[df["year"].between(lo, hi)]

    # Build a fresh HoverTool on every call: Bokeh models must not be
    # shared between browser sessions in a Panel server.
    hover = HoverTool(
        tooltips="""<div style="font:12px Inter,Helvetica,sans-serif;padding:2px 4px">
                      <b>@country</b><br>@year: @emissions{0,0} MtCO₂e</div>""",
        line_policy="nearest", mode="mouse")

    others = [c for c in COUNTRIES if c not in highlighted]
    layers = []
    for c in others + list(highlighted):            # grey first, colors on top
        d = d_all[d_all["country"] == c]
        if d.empty:
            continue
        hl = c in highlighted
        color = COLORS[c] if hl else GREY
        layers.append(hv.Curve(d, "year", ["emissions", "country"]).opts(
            color=color, line_width=2.5 if hl else 1.5,
            hover_line_color=color if hl else GREY_HOVER, hover_line_width=4,
            tools=[hover]))
        if hl:
            layers.append(hv.Text(d["year"].iloc[-1] + 0.4, d["emissions"].iloc[-1], c).opts(
                text_align="left", text_baseline="middle",
                text_color=color, text_font_size="10pt"))

    span = hi - lo
    return hv.Overlay(layers).opts(
        title=f"Greenhouse gas emissions, {lo}–{hi}",
        responsive=True, height=480,
        xlim=(lo, hi + max(2, span * 0.15)),        # room for end labels
        ylim=(None, None) if log else (0, None),
        logy=log,
        xlabel="", ylabel="MtCO₂e",
        show_legend=False, show_grid=True,
        gridstyle={"xgrid_line_alpha": 0, "ygrid_line_color": "#f1f5f9"},
        toolbar=None, hooks=[_minimal])


chart_pane = pn.pane.HoloViews(pn.bind(make_chart, highlight, years, log_y))

# Reusable component: put `ghg_card` into any other Panel layout or page.
ghg_card = pn.Column(
    chart_pane,
    pn.pane.Markdown(
        "Source: Our World in Data (total GHG excluding land use). "
        "Hover a line for values.",
        styles={"color": "#64748b", "font-size": "12px"}),
)

# ---------------------------------------------------------------- app
pn.template.FastListTemplate(
    title="Emissions dashboard",
    sidebar=[highlight, years, pn.Row(pn.widgets.StaticText(value="Log scale"), log_y)],
    main=[ghg_card],
    accent="#2563eb",
    header_background="#0f172a",
).servable()


FastListTemplate
    [js_area] HTML(None, height=0, margin=0, sizing_mode='fixed', width=0)
    [actions] TemplateActions()
    [browser_info] BrowserInfo()
    [busy_indicator] LoadingSpinner(height=20, width=20)
    [main-2440339206032] Column(sizing_mode='stretch_width')
        [0] HoloViews(Overlay, height=480, sizing_mode='stretch_width')
        [1] Markdown(str, sizing_mode='stretch_width', styles={'color': '#64748b', ...})
    [nav-2440339197296] MultiChoice(label='Highlight countries', max_items=4, name='Highlight countries', options=['United States', ...], sizing_mode='stretch_width', value=['United States', ...])
    [nav-2440339197632] IntRangeSlider(end=2024, label='Years', name='Years', sizing_mode='stretch_width', start=1990, value=(1990, 2024), value_end=2024, value_start=1990)
    [nav-2441493349072] Row(sizing_mode='stretch_width')
        [0] StaticText(sizing_mode='stretch_width', value='Log scale')
        [1] Switch(label='Log scale', name='Log scale', sizing_mode='stretch_width')

### AI Log

**Entry X:** 
- **Tool**: Claude
- **What I asked for**:
- **What came back**: 
- **What was wrong or could be better**: 
- **How I noticed**:
- **What I did**:

**Entry 1:** Summarize the menu of suggested libraries
- **Tool**: Claude (Sonnet 5.5)
- **What I asked for**: The instructions for this assignment provide a laundry list of Python libraries, some for plotting, some for the widgets/controls to build the dashboard, and some for hosting web apps. I'm familiar with some of these libraries but was unsure which are the most straightforward for integrating into Jupyter. I asked Claude to introduce the libraries, provide URLs for them (including for plotting library galleries), and explain which work well with Jupyter.
- **What came back**: Claude identified Panel/HoloViews, Dash, Gradio, Solara, and Voilà as working in notebooks. It said that Streamlit, Shiny, marimo, and Observable aren't usually run inside Jupyter. It noted that Render offers free hosting for web apps.
- **What was wrong or could be better**: Nothing 
- **How I noticed**: I checked the Panel and HoloView sites and looked through their galleries. Many of the examples are provided as Jupyter notebooks and seemed to be similar to the use case for this assignment. Panel/HoloView also offer flexibility with plotting libraries, so I could create some charts with hvPlot, some with seaborn, and some with Plotly. 
- **What I did**: I decided to go with Panel/Holoview.


